In [1]:
import pydough

#%load_ext pydough.jupyter_extensions
%reload_ext pydough.jupyter_extensions

#Necessary for comparison
import pandas as pd
from pandas.testing import assert_frame_equal, assert_series_equal
import re
import eval
import datetime

import collections
import numpy as np
import sqlite3 as sql
import os

In [52]:
#YOUR .SQL FILE TO CREATE THE DATABASE, COPY IT TO THIS FOLDER.
SQL_path = 'databases/Defog/init_defog.sql'

#METADATA FOR THE GRAPH .JSON
metadata_path = "metadata/Defog/DermTreatment_graph.json"

#GRAPH NAME
graph_name = "DermTreatment"

#DESIRED DATABASE NAME
DB_name = "notebookTest.db"



with open(SQL_path, 'r') as sql_file:
    sql_script = sql_file.read()

os.remove(DB_name)
connection = sql.connect(DB_name)
cursor = connection.cursor()
cursor.executescript(sql_script)

pydough.active_session.load_metadata_graph(metadata_path, graph_name)
pydough.active_session.connect_database("sqlite", database=DB_name)

DatabaseContext(connection=<pydough.database_connectors.database_connector.DatabaseConnection object at 0x750cf23f9700>, dialect=<DatabaseDialect.SQLITE: 'sqlite'>)

In [1]:
tested_file, tested_df = eval.compare_output("evalNotebookFiles/", "evalNotebookFiles/noMatch.csv", ".", ".")

AttributeError: 'builtin_function_or_method' object has no attribute 'compare_output'

In [ ]:
query = '''
SELECT d.drug_name, COUNT(*) AS num_treatments, AVG(t.tot_drug_amt) AS avg_drug_amt FROM treatments AS t JOIN drugs AS d ON t.drug_id = d.drug_id GROUP BY d.drug_name ORDER BY CASE WHEN num_treatments IS NULL THEN 1 ELSE 0 END DESC, num_treatments DESC, CASE WHEN avg_drug_amt IS NULL THEN 1 ELSE 0 END DESC, avg_drug_amt DESC, d.drug_name DESC, LIMIT 10;'''

sql_output = pd.read_sql_query(query, connection)
sql_output

,drug_name,num_treatments,avg_drug_amt
0,Drugalin,6,206.666667
1,Medicol,6,178.333333
2,Topizol,4,307.500000
3,Biologic-X,4,150.000000
4,Topicort,3,580.000000
5,Smallazine,3,203.333333


In [53]:
%%pydough

result = Drugs.CALCULATE(
    drug_name=drug_name,
    num_treatments=COUNT(treatments_used_in),
    avg_drug_amount=AVG(treatments_used_in.tot_drug_amt)
).ORDER_BY(
    num_treatments.DESC(),
    drug_name.ASC()
).TOP_K(5, by=num_treatments.DESC())

result = pydough.to_df(result)
print(result)

    drug_name  num_treatments  avg_drug_amount
0    Drugalin               6       206.666667
1     Medicol               6       178.333333
2     Topizol               4       307.500000
3  Biologic-X               4       150.000000
4  Smallazine               3       203.333333


In [36]:
%%pydough


selected_sessions = sessions.WHERE(
        (session_start_ts >= "2023-06-01") & (session_end_ts < "2023-06-08")
    ).CALCULATE(duration=DATEDIFF("seconds", session_start_ts, session_end_ts))
result = (
    Users.WHERE(HAS(selected_sessions))
    .CALCULATE(uid=uid, total_duration=SUM(selected_sessions.duration))
    .ORDER_BY(total_duration.DESC())
)
result = pydough.to_df(result)
print(result)

DATEDIFF unsupported for 'DAYS'.


    uid  total_duration
0     6            4205
1     2            3034
2     4            2736
3     8            2735
4     1            2715
5     1            2113
6     5            2098
7    10            1843
8     9            1797
9     7            1518
10    8             905
11    3             748
12    3             622
13    1             190
14    2             190
